In [18]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

In [19]:
FEATURE_NAMES = ("R", "W", "D", "S", "P", "O")

BOUNDS = {
    "R": (1400.0, 2000.0),
    "W": (40.0, 100.0),
    "D": (10.0, 30.0),
    "P": (0.03, 0.18),
}
CATEGORIES = {"S": ("square", "round"), "O": ("O1", "O2")}

# Hidden optimum per feature in [0, 1] (categoricals: 0 = first level, 1 = second).
_OPT = {"R": 0.65, "W": 0.35, "D": 0.55, "S": 1.0, "P": 0.45, "O": 0.0}


def make_feature_importance(
    active_features: list[str] | None = None,
) -> dict[str, float]:
    """Weight 1.0 for active features and 0.0 for the rest (None = all active)."""
    if active_features is None:
        return {name: 1.0 for name in FEATURE_NAMES}
    unknown = set(active_features) - set(FEATURE_NAMES)
    if unknown:
        raise ValueError(f"Unknown feature(s): {sorted(unknown)}")
    return {name: float(name in active_features) for name in FEATURE_NAMES}


def _normalize(name: str, value: Any) -> float:
    if name in CATEGORIES:
        if value not in CATEGORIES[name]:
            raise ValueError(f"{name} must be one of {CATEGORIES[name]}, got {value!r}")
        return float(CATEGORIES[name].index(value))
    lo, hi = BOUNDS[name]
    if not lo <= float(value) <= hi:
        raise ValueError(f"{name}={value} outside [{lo}, {hi}]")
    return (float(value) - lo) / (hi - lo)


def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + np.exp(-x))


def mock_welding_experiment(
    R: float,
    W: float,
    D: float,
    S: str,
    P: float,
    O: str,
    feature_importance: dict[str, float] | None = None,
    n_regions: int = 3,
    n_measurements: int = 5,
    sigma_p_noise_std: float = 8.0,
    uts_noise_std: float = 5.0,
    visual_noise_std: float = 0.4,
    seed: int | None = None,
) -> dict[str, Any]:
    """Mock welding response with controllable feature importance and noise.

    A feature with weight 0.0 has no effect on any output, e.g.
    feature_importance={"R": 1.0} makes outputs depend only on R.
    Sigma is measured in n_regions regions, n_measurements times each, returned as
    flat keys sigma_R{r}_M{m} (1-based). Also returns n_measurements values for uts
    and one visual label.
    """
    if n_regions < 1 or n_measurements < 1:
        raise ValueError("n_regions and n_measurements must be >= 1")

    rng = np.random.default_rng(seed)
    raw = {"R": R, "W": W, "D": D, "S": S, "P": P, "O": O}
    weights = (
        make_feature_importance()
        if feature_importance is None
        else {n: float(feature_importance.get(n, 0.0)) for n in FEATURE_NAMES}
    )

    # Squared distance to the hidden optimum, per feature, weighted by importance.
    dist = {n: (_normalize(n, raw[n]) - _OPT[n]) ** 2 for n in FEATURE_NAMES}
    penalty = sum(weights[n] * dist[n] for n in FEATURE_NAMES)

    # Lower residual stress, higher UTS and better appearance near the optimum.
    sigma_p_mean = 120.0 + 300.0 * penalty
    uts_mean = 260.0 - 120.0 * penalty
    visual_score = 2.0 - 4.0 * penalty

    # Each region has its own small systematic offset around the mean.
    region_offsets = rng.normal(0.0, sigma_p_noise_std, size=n_regions)
    sigma_p = (
        sigma_p_mean
        + region_offsets[:, None]
        + rng.normal(0.0, sigma_p_noise_std, size=(n_regions, n_measurements))
    )
    uts = uts_mean + rng.normal(0.0, uts_noise_std, size=n_measurements)
    visual = bool(
        rng.random() < _sigmoid(visual_score + rng.normal(0.0, visual_noise_std))
    )

    result: dict[str, Any] = {
        f"sigma_R{r + 1}_M{m + 1}": float(sigma_p[r, m])
        for r in range(n_regions)
        for m in range(n_measurements)
    }
    result["uts"] = [float(x) for x in uts]
    result["visual_quality"] = visual
    return result

In [20]:
mock_welding_experiment(
    R=1700,
    W=60,
    D=20,
    S="round",
    P=0.1,
    O="O1",
    feature_importance={"R": 1.0},
    n_regions=3,
    n_measurements=5,
    seed=0,
)

{'sigma_R1_M1': 128.59504270597145,
 'sigma_R1_M2': 123.47048678345826,
 'sigma_R1_M3': 130.64860220802302,
 'sigma_R1_M4': 138.18784212978824,
 'sigma_R1_M5': 135.3324894737811,
 'sigma_R2_M1': 120.06327920721364,
 'sigma_R2_M2': 115.56978932530117,
 'sigma_R2_M3': 120.70696539337077,
 'sigma_R2_M4': 126.02376892844754,
 'sigma_R2_M5': 107.09291489655891,
 'sigma_R3_M1': 130.12304789208588,
 'sigma_R3_M2': 121.90609362552172,
 'sigma_R3_M3': 126.01524236591864,
 'sigma_R3_M4': 127.51930934068777,
 'sigma_R3_M5': 129.342979952593,
 'uts': [259.35815268187065,
  262.5125668472134,
  256.6573266852798,
  264.13231735274843,
  253.97402663256693],
 'visual_quality': True}

In [21]:
folder_path = Path(os.getcwd()) / "data"

In [22]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)

,trial_index,R,W,D,S,P,O,visual_quality,sigma_R1_M1,sigma_R1_M2,...,sigma_R2_M2,sigma_R2_M3,sigma_R2_M4,sigma_R2_M5,sigma_R3_M1,sigma_R3_M2,sigma_R3_M3,sigma_R3_M4,sigma_R3_M5,uts
0,0,1700,70,20,round,0.105000,O2,1,128.595043,123.470487,...,115.569789,120.706965,126.023769,107.092915,130.123048,121.906094,126.015242,127.519309,129.342980,259.358153
1,1,1462,81,25,square,0.162470,O1,0,201.992749,219.660854,...,218.579338,216.453656,220.599982,210.334646,210.993550,208.439875,217.087600,212.614607,209.957176,220.229124
2,2,1733,44,18,round,0.102709,O2,1,104.688188,138.617586,...,114.094930,126.346052,116.041060,115.894921,113.065818,123.042656,118.609407,123.765301,114.545506,259.551139
3,3,1997,92,21,round,0.136314,O2,1,167.492696,168.413659,...,161.846176,137.068473,132.441133,133.011880,153.707920,150.611086,155.925883,162.907854,157.143862,250.505794
4,4,1649,63,13,square,0.055458,O1,1,136.626353,118.221992,...,137.103937,137.052809,147.774770,137.702922,153.961665,137.932358,167.899125,134.552127,158.691707,251.723480


In [23]:
print(exp.columns)
cols_output = [
    "visual_quality",
    "uts",
]

cols_R1 = [
    "sigma_R1_M1",
    "sigma_R1_M2",
    "sigma_R1_M3",
    "sigma_R1_M4",
    "sigma_R1_M5",
]

cols_R2 = [
    "sigma_R2_M1",
    "sigma_R2_M2",
    "sigma_R2_M3",
    "sigma_R2_M4",
    "sigma_R2_M5",
]

cols_R3 = [
    "sigma_R3_M1",
    "sigma_R3_M2",
    "sigma_R3_M3",
    "sigma_R3_M4",
    "sigma_R3_M5",
]

Index(['trial_index', 'R', 'W', 'D', 'S', 'P', 'O', 'visual_quality',
       'sigma_R1_M1', 'sigma_R1_M2', 'sigma_R1_M3', 'sigma_R1_M4',
       'sigma_R1_M5', 'sigma_R2_M1', 'sigma_R2_M2', 'sigma_R2_M3',
       'sigma_R2_M4', 'sigma_R2_M5', 'sigma_R3_M1', 'sigma_R3_M2',
       'sigma_R3_M3', 'sigma_R3_M4', 'sigma_R3_M5', 'uts'],
      dtype='str')


In [24]:
filt_R1 = exp[cols_R1].isna().all(axis=1)
filt_R2 = exp[cols_R2].isna().all(axis=1)
filt_R3 = exp[cols_R3].isna().all(axis=1)
filt_output = exp[cols_output].isna().any(axis=1)
filt_removed = filt_R1 | filt_R2 | filt_R3 | filt_output
filt_removed

0    False
1    False
2    False
3    False
4    False
dtype: bool

In [25]:
exp_todo = exp[filt_removed].copy()
exp
display(exp)
display(exp_todo)

,trial_index,R,W,D,S,P,O,visual_quality,sigma_R1_M1,sigma_R1_M2,...,sigma_R2_M2,sigma_R2_M3,sigma_R2_M4,sigma_R2_M5,sigma_R3_M1,sigma_R3_M2,sigma_R3_M3,sigma_R3_M4,sigma_R3_M5,uts
0,0,1700,70,20,round,0.105000,O2,1,128.595043,123.470487,...,115.569789,120.706965,126.023769,107.092915,130.123048,121.906094,126.015242,127.519309,129.342980,259.358153
1,1,1462,81,25,square,0.162470,O1,0,201.992749,219.660854,...,218.579338,216.453656,220.599982,210.334646,210.993550,208.439875,217.087600,212.614607,209.957176,220.229124
2,2,1733,44,18,round,0.102709,O2,1,104.688188,138.617586,...,114.094930,126.346052,116.041060,115.894921,113.065818,123.042656,118.609407,123.765301,114.545506,259.551139
3,3,1997,92,21,round,0.136314,O2,1,167.492696,168.413659,...,161.846176,137.068473,132.441133,133.011880,153.707920,150.611086,155.925883,162.907854,157.143862,250.505794
4,4,1649,63,13,square,0.055458,O1,1,136.626353,118.221992,...,137.103937,137.052809,147.774770,137.702922,153.961665,137.932358,167.899125,134.552127,158.691707,251.723480


,trial_index,R,W,D,S,P,O,visual_quality,sigma_R1_M1,sigma_R1_M2,...,sigma_R2_M2,sigma_R2_M3,sigma_R2_M4,sigma_R2_M5,sigma_R3_M1,sigma_R3_M2,sigma_R3_M3,sigma_R3_M4,sigma_R3_M5,uts


In [26]:
SIGMA_COLUMNS = [f"sigma_R{r}_M{m}" for r in range(1, 4) for m in range(1, 6)]

for idx, row in exp_todo.iterrows():
    result = mock_welding_experiment(
        R=row["R"],
        W=row["W"],
        D=row["D"],
        S=row["S"],
        P=row["P"],
        O=row["O"],
        feature_importance={"R": 1.0},
        n_regions=3,
        n_measurements=5,
        seed=idx,
    )
    for col in SIGMA_COLUMNS:
        exp.loc[idx, col] = result[col]
    exp.loc[idx, "visual_quality"] = float(result["visual_quality"])
    exp.loc[idx, "uts"] = result["uts"][0]


In [27]:
exp.to_excel(folder_path / "experiments.xlsx", index=False)